In [ ]:
"""
MscaleDNN-PINN — 二维 Poisson 方程求解
========================================
PDE:   -Δu(x,y) = f(x,y),   (x,y) ∈ Ω = [-1,1]²
边界:   u = F(x,y),          (x,y) ∈ ∂Ω   (Dirichlet)

精确解: F(x,y) = g(x) + g(y),  g(x) = exp(-x²) sin(30 x²)
源项:   f(x,y) = -(g''(x) + g''(y))

损失函数:
    L = λ_r · MSE[ -Δu - f ]  +  λ_b · MSE[ u - F ]

网络:  MscaleDNN（多尺度子网并行 → 融合层 → 标量输出）
训练:  15000 epoch, Adam(lr=1e-3) + CosineAnnealingLR
"""

import os
import torch
import torch.nn as nn
import numpy as np
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
torch.manual_seed(42)
np.random.seed(42)

CHECKPOINT_DIR = "checkpoints_mscalednn_pinn_poisson"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MU = 30.0
LAMBDA_R = 1.0
LAMBDA_B = 10.0


# ==============================================================================
# 精确解 & 源项
# ==============================================================================

def g_func(x):
    return torch.exp(-x ** 2) * torch.sin(MU * x ** 2)

def g_second(x):
    ex = torch.exp(-x ** 2)
    s  = torch.sin(MU * x ** 2)
    c  = torch.cos(MU * x ** 2)
    coeff_s = 4 * x**2 - 4 * MU**2 * x**2 - 2
    coeff_c = 2 * MU - 8 * MU * x**2
    return ex * (coeff_s * s + coeff_c * c)

def F_exact(xy):
    return g_func(xy[:, 0:1]) + g_func(xy[:, 1:2])

def f_source(xy):
    """右端项 f = -(g''(x) + g''(y))，配套 -Δu = f"""
    return -(g_second(xy[:, 0:1]) + g_second(xy[:, 1:2]))

def g_boundary(xy):
    return F_exact(xy)


# ==============================================================================
# MscaleDNN 网络结构
# ==============================================================================

class ScaleSubNet(nn.Module):
    def __init__(self, scale_factor, hidden_dim, num_layers):
        super().__init__()
        self.scale = scale_factor
        layers = []
        in_dim = 2
        for _ in range(num_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.layers = nn.ModuleList(layers)
        for m in self.layers:
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)

    def forward(self, xy):
        x = xy * self.scale
        n = len(self.layers)
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < n - 1:
                x = torch.sin(x) if i % 2 == 0 else torch.tanh(x)
        return x

class MscaleDNN(nn.Module):
    def __init__(self, scales=None, hidden_dim=64, sub_layers=4):
        super().__init__()
        if scales is None:
            scales = [1, 2, 4, 8, 16, 32]
        self.subnets = nn.ModuleList([
            ScaleSubNet(s, hidden_dim, sub_layers) for s in scales
        ])
        fuse_in = hidden_dim * len(scales)
        self.fuse = nn.Sequential(
            nn.Linear(fuse_in, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.xavier_normal_(self.fuse[0].weight); nn.init.zeros_(self.fuse[0].bias)
        nn.init.xavier_normal_(self.fuse[2].weight); nn.init.zeros_(self.fuse[2].bias)

    def forward(self, xy):
        return self.fuse(torch.cat([s(xy) for s in self.subnets], dim=-1))


# ==============================================================================
# 自动微分：计算 Δu
# 关键：每次调用都在函数内部对传入的 xy 重新 requires_grad_(True)，
#       确保每步计算图干净，不会跨 backward 累积梯度。
# ==============================================================================

def laplacian(model, xy):
    """
    xy: (N,2) 普通张量（不需要在外部设 requires_grad）
    函数内部 clone + requires_grad_(True)，每步计算图完全独立。
    """
    xy = xy.detach().requires_grad_(True)   # ← 关键：每次创建新的计算图入口
    u  = model(xy)

    grad_u = torch.autograd.grad(
        u, xy, grad_outputs=torch.ones_like(u),
        create_graph=True, retain_graph=True
    )[0]

    u_xx = torch.autograd.grad(
        grad_u[:, 0:1], xy, grad_outputs=torch.ones_like(grad_u[:, 0:1]),
        create_graph=True, retain_graph=True
    )[0][:, 0:1]

    u_yy = torch.autograd.grad(
        grad_u[:, 1:2], xy, grad_outputs=torch.ones_like(grad_u[:, 1:2]),
        create_graph=True, retain_graph=True
    )[0][:, 1:2]

    return u_xx + u_yy


# ==============================================================================
# 采样（网格 + 随机混合，与 Document 4 一致）
# ==============================================================================

def sample_interior(n):
    side   = int(np.sqrt(n // 2))
    gx     = torch.linspace(-1 + 1e-4, 1 - 1e-4, side, dtype=torch.float64)
    gy     = torch.linspace(-1 + 1e-4, 1 - 1e-4, side, dtype=torch.float64)
    GX, GY = torch.meshgrid(gx, gy, indexing='ij')
    xy_uni = torch.stack([GX.flatten(), GY.flatten()], dim=-1)
    n_rand = n - xy_uni.shape[0]
    xy_rnd = torch.rand(n_rand, 2, dtype=torch.float64) * (2 - 2e-4) - (1 - 1e-4)
    return torch.cat([xy_uni, xy_rnd], dim=0).to(device)

def sample_boundary(n_per_edge=200):
    t    = torch.linspace(-1, 1, n_per_edge, dtype=torch.float64)
    bot  = torch.stack([t,  -torch.ones_like(t)], dim=-1)
    top  = torch.stack([t,   torch.ones_like(t)], dim=-1)
    left = torch.stack([-torch.ones_like(t), t],  dim=-1)
    rgt  = torch.stack([ torch.ones_like(t), t],  dim=-1)
    return torch.cat([bot, top, left, rgt], dim=0).to(device)


# ==============================================================================
# 训练
# ==============================================================================

def train_mscalednn_pinn(
    scales       = None,
    hidden_dim   = 150,
    sub_layers   = 4,
    n_interior   = 10000,
    n_per_edge   = 200,
    total_epochs = 15000,
    lr           = 2.5e-3,
    log_every    = 1,
    lambda_r     = LAMBDA_R,
    lambda_b     = LAMBDA_B,
):
    if scales is None:
        scales = [1, 2, 4, 8, 16, 32]

    # 数据（xy_int 存为普通张量，requires_grad 在 laplacian 内部管理）
    xy_int = sample_interior(n_interior)   # 普通张量，不设 requires_grad
    xy_bc  = sample_boundary(n_per_edge)
    u_bc   = g_boundary(xy_bc)

    # 测试集
    nx = 100
    xv, yv  = np.meshgrid(np.linspace(-1, 1, nx), np.linspace(-1, 1, nx))
    xy_test = torch.tensor(np.stack([xv.ravel(), yv.ravel()], axis=1),
                           dtype=torch.float64, device=device)
    u_test       = F_exact(xy_test).detach()
    u_test_sq_mean = torch.mean(u_test ** 2).item()

    model = MscaleDNN(scales, hidden_dim, sub_layers).double().to(device)
    print(f"参数量: {sum(p.numel() for p in model.parameters()):,}  scales: {scales}")

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    # sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, total_epochs, eta_min=1e-5)

    epochs_record, res_losses, bc_losses, total_losses, l2_errors = [], [], [], [], []
    t0 = time.time()

    for ep in range(1, total_epochs + 1):
        model.train()
        opt.zero_grad()

        # PDE 残差：-Δu - f = 0
        lap_u    = laplacian(model, xy_int)          # xy_int 传入普通张量即可
        loss_r   = torch.mean((-lap_u - f_source(xy_int.detach())) ** 2)

        # 边界损失
        loss_b   = nn.functional.mse_loss(model(xy_bc), u_bc)

        loss = lambda_r * loss_r + lambda_b * loss_b
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        # sch.step()

        # if ep % log_every == 0 or ep == 1:
        if ep % log_every == 0 or ep == 1:
            with torch.no_grad():
                pred_test = model(xy_test)
                l2 = torch.sqrt(torch.mean((pred_test - u_test)**2) / u_test_sq_mean).item()
            epochs_record.append(ep)
            res_losses.append(loss_r.item())
            bc_losses.append(loss_b.item())
            total_losses.append(loss.item())
            l2_errors.append(l2)
            print(f"ep {ep:6d} | res: {loss_r.item():.3e} | bc: {loss_b.item():.3e} | "
                  f"total: {loss.item():.3e} | L2: {l2:.6f} | {time.time()-t0:.1f}s")

    with torch.no_grad():
        fp = model(xy_test)
        fl2  = torch.sqrt(torch.mean((fp - u_test)**2) / u_test_sq_mean).item()
        fmae = torch.mean(torch.abs(fp - u_test)).item()
    print(f"\n训练完成 | L2: {fl2:.6f} | MAE: {fmae:.6f} | 时间: {time.time()-t0:.1f}s")

    torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "mscalednn_pinn_weights.pt"))
    np.savez(os.path.join(CHECKPOINT_DIR, "mscalednn_pinn_curves.npz"),
             epochs=np.array(epochs_record), res_loss=np.array(res_losses),
             bc_loss=np.array(bc_losses), total_loss=np.array(total_losses),
             l2_error=np.array(l2_errors))
    print(f"已保存至 {CHECKPOINT_DIR}/")


# ==============================================================================
if __name__ == "__main__":
    train_mscalednn_pinn()